## Smoke test

The Self-ReSET capacity trial (`raysubmit_RT83NPQcRj7Teqve`, 2026-11-17 PST) runs three rollouts, not a full epoch or paper replication. It starts fresh from Qwen3-8B and its converted Megatron reference checkpoint on one RunPod NVIDIA H100 80 GB HBM3 with a 234 GiB host-memory limit. The stack is `slimerl/slime:nightly-dev-20260430b`, Python 3.12.3, PyTorch 2.9.1+cu129, CUDA 12.9, Megatron-LM `1dcf0dafa`, Ray 2.55.1, and SGLang 0.5.10.post1 with Transformers 5.3.0. Training uses BF16 and seed 1234; the trainer and rollout engine share the GPU. GRPO uses the Self-ReSET safety reward, partial rollouts and masking, and rejects prompt groups with zero reward variance. Each rollout targets 4 shuffled prompts x 4 responses (global batch 16; oversampling batch 4) from `vanilla_filter_balanced_1500.jsonl` (3,000 records), with a 512-new-token cap, temperature 0.7, top-p 1.0, top-k disabled, and repetition penalty 1.0. Adam uses constant LR 1e-6, betas (0.9, 0.98), weight decay 0.1, gradient clipping 1.0, GRPO clip bounds 0.20/0.28, and zero KL and entropy coefficients. The trainer uses dynamic batching (2,048 tokens/GPU), full activation recomputation, and CPU optimizer offload while keeping the model on GPU. SGLang uses a 0.45 static memory fraction, a 4,096-token cache cap, and both ordinary and piecewise CUDA graphs disabled. Qwen3Guard-Stream-8B, Qwen3Guard-Gen-8B, and Llama-Guard-3-8B run on CPU under Transformers 4.55.0. Evaluation is configured before training and after the final rollout on 250 prompts each from WildJailbreak and XSTest, with two responses per prompt, a 512-new-token cap, temperature 0, top-p 1.0, top-k disabled, and repetition penalty 1.2. The [launcher](slime/scripts/runpod.sh) reuses the Ray head on port 8266 and saves after each rollout, retaining only the latest checkpoint in `/workspace/checkpoints/Self-RESET-trial-20261118T013141Z-41887`. At the 17:44 PST check, the initial evaluation was still generating and no optimizer step had been confirmed.

Original paper: [Self-ReSET (arXiv:2605.08936)](https://arxiv.org/abs/2605.08936).

Launch from the repository root on the pod:

```bash
RUNPOD_REUSE_RAY=YES RAY_DASHBOARD_PORT=8266 \
  RUNPOD_TRAIN_TRIAL=YES CONFIRM_RUN=YES \
  bash replication/slime/scripts/runpod.sh
```


### 1. Initial evaluation (before training)

- [x] Generate **1,000 responses** with Qwen3-8B: 250 WildJailbreak prompts + 250 XSTest prompts, 2 responses per prompt, up to 512 new tokens each.
- [x] Score all **1,000 responses** on CPU: **Llama-Guard-3-8B** labels safety; **Qwen3Guard-Gen-8B** labels refusal. The first five completed responses per dataset also get extra guard checks for logging.
- [x] Log the baseline evaluation metrics.

### 2. Training (repeat 3 times)

For each rollout, generate 4 responses per prompt and collect 4 accepted prompt groups for a **training batch of 16 responses**. Qwen3Guard-Gen-8B scores candidates for the reward; groups with no reward variation are rejected, so generation can exceed 16 responses per rollout. Qwen3Guard-Stream-8B labels eligible responses at the token level for partial-rollout processing. Each rollout ends with a GRPO optimizer update, checkpoint save, and weight sync. Only the latest checkpoint is retained.

- [ ] Complete rollout 1.
- [ ] Complete rollout 2.
- [ ] Complete rollout 3.

> As of 8:37 PM PST on November 17, 2026: baseline evaluation complete; no rollout or checkpoint confirmed yet.

### 3. Final evaluation (after rollout 3)

- [ ] Generate and score **another 1,000 responses** on the same two datasets with the same guard models.
- [ ] Log the final evaluation metrics.
- [ ] Confirm the RunPod pod stopped. Auto-stop is armed for job completion or failure.